<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/varshini/Module_2_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [90]:
!pip install streamlit streamlit-webrtc av pydub openai-whisper python-dotenv -q
!pip install pyngrok -q


In [91]:
from google.colab import userdata
import os

# Get ngrok auth token from Colab Secrets (secure method)
ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')

if not ngrok_auth_token:
    print("⚠️ ngrok auth token not found!")
    print("Please add it to Colab Secrets:")
    print("1. Click '🔑' icon on left panel")
    print("2. Add new secret named 'NGROK_AUTH_TOKEN'")
    print("3. Paste your token from https://dashboard.ngrok.com/auth")
    raise ValueError("NGROK_AUTH_TOKEN not configured")

# Set ngrok auth
os.environ['NGROK_AUTHTOKEN'] = ngrok_auth_token
print("✅ ngrok configured securely from Colab Secrets")


✅ ngrok configured securely from Colab Secrets


In [92]:
# Create the streamlit app file
streamlit_code = '''
import streamlit as st
from stt_engine import STTEngine
import numpy as np
import os


st.set_page_config(page_title="Live Speech-to-Text", page_icon="🎙️")
st.title("🎙️ Live Speech-to-Text")

st.info("🎤 Record audio directly in your browser")

# Load STT Engine
if "stt_engine" not in st.session_state:
    try:
        with st.spinner("Loading Whisper Model (this may take a minute)..."):
            st.session_state["stt_engine"] = STTEngine(model_size="base")
        st.success("✅ Whisper STT Engine Loaded!")
    except Exception as e:
        st.error(f"Failed to load STT Engine: {e}")
        st.stop()


# HTML + JavaScript for browser audio recording
st.markdown("---")
st.markdown("### 🎙️ Record Audio")

html_code = """
<html>
<head>
    <style>
        .audio-controls {
            display: flex;
            gap: 10px;
            margin: 20px 0;
            flex-wrap: wrap;
        }
        button {
            padding: 10px 20px;
            font-size: 16px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            transition: background-color 0.3s;
        }
        .start-btn {
            background-color: #4CAF50;
            color: white;
        }
        .stop-btn {
            background-color: #f44336;
            color: white;
        }
        .download-btn {
            background-color: #2196F3;
            color: white;
        }
        .status {
            margin-top: 20px;
            padding: 10px;
            border-radius: 5px;
            font-weight: bold;
        }
        .recording {
            background-color: #ffebee;
            color: #c62828;
        }
        .stopped {
            background-color: #e8f5e9;
            color: #2e7d32;
        }
    </style>
</head>
<body>
    <div class="audio-controls">
        <button class="start-btn" onclick="startRecording()">🎙️ START</button>
        <button class="stop-btn" onclick="stopRecording()" disabled id="stopBtn">⏹️ STOP</button>
        <button class="download-btn" onclick="downloadAudio()" disabled id="downloadBtn">⬇️ DOWNLOAD</button>
    </div>

    <div id="status" class="status stopped" style="display:none;"></div>
    <div id="audioPlayer"></div>

    <script>
        let mediaRecorder;
        let audioChunks = [];

        async function startRecording() {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
                mediaRecorder = new MediaRecorder(stream);
                audioChunks = [];

                mediaRecorder.ondataavailable = (event) => {
                    audioChunks.push(event.data);
                };

                mediaRecorder.onstop = () => {
                    const audioBlob = new Blob(audioChunks, { type: 'audio/wav' });
                    const audioUrl = URL.createObjectURL(audioBlob);
                    const audioPlayer = document.getElementById('audioPlayer');
                    audioPlayer.innerHTML = '<audio controls style="width:100%;"><source src="' + audioUrl + '" type="audio/wav"></audio>';
                    document.getElementById('downloadBtn').disabled = false;
                    window.recordedAudio = audioBlob;
                };

                mediaRecorder.start();
                document.querySelector('.start-btn').disabled = false;
                document.getElementById('stopBtn').disabled = false;

                const statusDiv = document.getElementById('status');
                statusDiv.style.display = 'block';
                statusDiv.textContent = '🔴 RECORDING...';
                statusDiv.className = 'status recording';

            } catch (error) {
                alert('Allow microphone access!');
            }
        }

        function stopRecording() {
            if (mediaRecorder) {
                mediaRecorder.stop();
                mediaRecorder.stream.getTracks().forEach(track => track.stop());
                document.querySelector('.start-btn').disabled = false;
                document.getElementById('stopBtn').disabled = true;

                const statusDiv = document.getElementById('status');
                statusDiv.textContent = '✅ Ready to upload!';
                statusDiv.className = 'status stopped';
            }
        }

        function downloadAudio() {
            if (window.recordedAudio) {
                const url = URL.createObjectURL(window.recordedAudio);
                const a = document.createElement('a');
                a.href = url;
                a.download = 'recording.wav';
                document.body.appendChild(a);
                a.click();
                document.body.removeChild(a);
            }
        }
    </script>
</body>
</html>
"""

st.components.v1.html(html_code, height=250)


st.markdown("---")
st.markdown("### 📤 Upload & Transcribe")

uploaded_file = st.file_uploader(
    "Upload your audio file",
    type=["wav", "mp3", "m4a", "flac", "ogg"]
)

if uploaded_file is not None:
    st.audio(uploaded_file)

    if st.button("✨ TRANSCRIBE", use_container_width=True):
        try:
            with st.spinner("⏳ Transcribing..."):
                temp_path = "temp_audio.wav"
                with open(temp_path, "wb") as f:
                    f.write(uploaded_file.getbuffer())

                transcript = st.session_state["stt_engine"].transcribe_file(temp_path)

                st.write("### 🎯 Transcription")
                if not transcript or "No speech detected" in transcript:
                    st.info("No speech detected in the audio.")
                else:
                    st.success(transcript)

            os.remove(temp_path) # Clean up the temporary file

        except Exception as e:
            st.error(f"Error during transcription: {e}")



'''

with open('app.py', 'w') as f:
    f.write(streamlit_code)

print("✅ app.py created")

✅ app.py created


In [93]:
stt_engine_code = '''
import whisper
import os

class STTEngine:
    def __init__(self, model_size="base"):
        """
        Initialize the STT Engine with Whisper model
        model_size: tiny, base, small, medium, large
        """
        self.model = whisper.load_model(model_size)

    def transcribe_file(self, audio_path):
        """
        Transcribe an audio file
        """
        try:
            if not os.path.exists(audio_path):
                return "Error: Audio file not found"

            result = self.model.transcribe(audio_path, language="en")
            return result.get("text", "No speech detected")
        except Exception as e:
            return f"Transcription error: {str(e)}"
'''

with open('stt_engine.py', 'w') as f:
    f.write(stt_engine_code)

print("✅ stt_engine.py created")


✅ stt_engine.py created


In [94]:
import os

# Create .streamlit directory
os.makedirs('.streamlit', exist_ok=True)

config_content = '''
[client]
showErrorDetails = true
allowRunOnSave = true

[server]
headless = true
port = 8501
enableXsrfProtection = false

[logger]
level = "info"
'''

with open('.streamlit/config.toml', 'w') as f:
    f.write(config_content)

print("✅ Streamlit config created")


✅ Streamlit config created


In [95]:
# Add this as a new cell
import subprocess
print("System audio devices:")
subprocess.run(['python', '-m', 'sounddevice'], capture_output=True)


System audio devices:


CompletedProcess(args=['python', '-m', 'sounddevice'], returncode=1, stdout=b'', stderr=b'/usr/bin/python3: No module named sounddevice\n')

In [96]:
# Debug: Check if microphone is accessible
import subprocess
result = subprocess.run(['python', '-c', '''
import sounddevice as sd
import numpy as np

print("Testing microphone...")
try:
    duration = 2
    fs = 44100
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()
    print(f"✅ Microphone works! Captured {len(recording)} samples")
except Exception as e:
    print(f"❌ Microphone error: {e}")
'''], capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)



STDERR: Traceback (most recent call last):
  File "<string>", line 2, in <module>
ModuleNotFoundError: No module named 'sounddevice'



In [97]:
import subprocess
import time
from pyngrok import ngrok

# Kill any existing streamlit processes
!pkill -f streamlit

# Kill any existing ngrok tunnels to prevent conflicts
ngrok.kill()

# Start Streamlit in background
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port=8501'])

# Give streamlit time to start
time.sleep(3)

# Create private ngrok tunnel
print("🔗 Creating private ngrok tunnel...")
public_url = ngrok.connect(8501, "http")
print(f"\n{'='*60}")
print(f"✅ STREAMLIT APP IS LIVE!")
print(f"{'='*60}")
print(f"\n🔐 Private Tunnel URL: {public_url}")
print(f"\nClick here to open: {public_url}")
print(f"\n{'='*60}")
print(f"⚠️  This URL is PRIVATE - only you can access it")
print(f"{'='*60}\n")

🔗 Creating private ngrok tunnel...

✅ STREAMLIT APP IS LIVE!

🔐 Private Tunnel URL: NgrokTunnel: "https://mistilled-unimported-milagros.ngrok-free.dev" -> "http://localhost:8501"

Click here to open: NgrokTunnel: "https://mistilled-unimported-milagros.ngrok-free.dev" -> "http://localhost:8501"

⚠️  This URL is PRIVATE - only you can access it

